# Civil Engineering Global Project Dataset

In [1]:

import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)


civil = pd.read_csv("Civil Engineering Global Project Dataset.csv")

print("Original dataset shape:", civil.shape)

target = "is_good"

civil[target] = pd.to_numeric(civil[target], errors="coerce")
civil = civil.dropna(subset=[target]).copy()

if not set(civil[target].unique()).issubset({0, 1}):
    raise ValueError("is_good must contain only 0 and 1.")

civil[target] = civil[target].astype(int)

print("\nOriginal class distribution:")
print(civil[target].value_counts(normalize=True).sort_index())


sample_size = 30_000

if len(civil) < sample_size:
    raise ValueError("Dataset contains fewer than 30,000 valid rows.")

civil, _ = train_test_split(
    civil,
    train_size=sample_size,
    stratify=civil[target],
    random_state=42
)

civil = civil.reset_index(drop=True)

civil.to_csv("civil_dataset_30k.csv", index=False)

print("\nReduced dataset shape:", civil.shape)
print("\nReduced class distribution:")
print(civil[target].value_counts())
print(civil[target].value_counts(normalize=True).round(4))


X = civil.drop(
    columns=[target, "Project ID"],
    errors="ignore"
)

y = civil[target]


numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)


# Numerical: fill missing values with median
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical: fill missing values and encode categories
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class proportions:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTesting class proportions:")
print(y_test.value_counts(normalize=True).round(4))


model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

print("\nModel training completed.")

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print("\n========== MODEL PERFORMANCE ==========")

print("Accuracy :",
      round(accuracy_score(y_test, y_pred), 4))

print("Precision:",
      round(precision_score(
          y_test, y_pred, zero_division=0
      ), 4))

print("Recall   :",
      round(recall_score(
          y_test, y_pred, zero_division=0
      ), 4))

print("F1-score :",
      round(f1_score(
          y_test, y_pred, zero_division=0
      ), 4))

if len(np.unique(y_test)) == 2:
    print("ROC-AUC  :",
          round(roc_auc_score(y_test, y_prob[:, 1]), 4))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

os.makedirs("saved_models", exist_ok=True)

model_path = "saved_models/civil_risk_model.pkl"
metadata_path = "saved_models/civil_risk_model_metadata.pkl"

joblib.dump(model, model_path)

metadata = {
    "target": target,
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "sample_size": len(civil),
    "training_rows": len(X_train),
    "test_rows": len(X_test),
    "sampling_method": "Stratified sampling",
    "random_state": 42
}

joblib.dump(metadata, metadata_path)

print("\nSaved files:")
print(model_path)
print(metadata_path)
print("Reduced dataset: civil_dataset_30k.csv")


loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]]

prediction = loaded_model.predict(sample)[0]
probabilities = loaded_model.predict_proba(sample)[0]

print("\n========== SAMPLE PREDICTION ==========")
print("Actual label:", y_test.iloc[0])
print("Predicted is_good:", prediction)
print("Probability of class 0:", round(probabilities[0], 4))
print("Probability of class 1:", round(probabilities[1], 4))

Original dataset shape: (1000000, 12)

Original class distribution:
is_good
0    0.49987
1    0.50013
Name: proportion, dtype: float64

Reduced dataset shape: (30000, 12)

Reduced class distribution:
is_good
1    15004
0    14996
Name: count, dtype: int64
is_good
1    0.5001
0    0.4999
Name: proportion, dtype: float64

Numeric features: ['Certificates', 'Years of Experience', 'age', 'Time Arrival Strafe', 'Project Cost', 'Project Proximity', 'Violation Risk Index', 'Company PCAB Score', 'Weekly Overtime Hours', 'Salary Bracket']
Categorical features: []

Training rows: 24000
Testing rows: 6000

Training class proportions:
is_good
1    0.5001
0    0.4999
Name: proportion, dtype: float64

Testing class proportions:
is_good
1    0.5002
0    0.4998
Name: proportion, dtype: float64

Model training completed.

========== MODEL PERFORMANCE ==========
Accuracy : 0.7918
Precision: 0.7936
Recall   : 0.7891
F1-score : 0.7913
ROC-AUC  : 0.7936

Classification Report:
              precision    re